In [ ]:
# 环境与参数（可编辑）
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
#@title 运行参数
OFFLINE = False  #@param {type:"boolean"}
MODEL_CACHE_DIR = ''  #@param {type:"string"}
PAPERS_DIR = 'papers'  #@param {type:"string"}
os.makedirs(PAPERS_DIR, exist_ok=True)

if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

if MODEL_CACHE_DIR:
    os.environ.setdefault('SENTENCE_TRANSFORMERS_HOME', MODEL_CACHE_DIR)
    os.environ.setdefault('HF_HOME', MODEL_CACHE_DIR)

print('OFFLINE =', OFFLINE)
print('MODEL_CACHE_DIR =', MODEL_CACHE_DIR)
print('PAPERS_DIR =', PAPERS_DIR)

In [ ]:
# DBLP 爬虫（对已有 txt 自动跳过）
import requests
from bs4 import BeautifulSoup

def generate_dblp_sources(conf_abbr, start_year, end_year):
    conf_lower = conf_abbr.lower()
    return [
        (f"{conf_abbr}{year}", f"https://dblp.org/db/conf/{conf_lower}/{conf_lower}{year}.html")
        for year in range(start_year, end_year + 1)
    ]

def get_sources():
    sources = [
      ('NIPS2019', 'https://dblp.org/db/conf/nips/nips2019.html'),
      ('NIPS2020', 'https://dblp.org/db/conf/nips/neurips2020.html'),
      ('NIPS2021', 'https://dblp.org/db/conf/nips/neurips2021.html'),
      ('NIPS2022', 'https://dblp.org/db/conf/nips/neurips2022.html'),
      ('NIPS2023', 'https://dblp.org/db/conf/nips/neurips2023.html'),
      ('NIPS2024', 'https://dblp.org/db/conf/nips/neurips2024.html'),
      ('ICCV2019', 'https://dblp.org/db/conf/iccv/iccv2019.html'),
      ('ICCV2021', 'https://dblp.org/db/conf/iccv/iccv2021.html'),
      ('ICCV2023', 'https://dblp.org/db/conf/iccv/iccv2023.html'),
      ('KDD2025', 'https://dblp.org/db/conf/kdd/kdd2025-1.html'),
      ('KDD2025', 'https://dblp.org/db/conf/kdd/kdd2025-2.html'),
      ('ACL2019', 'https://dblp.org/db/conf/acl/acl2019-1.html'),
      ('ACL2020', 'https://dblp.org/db/conf/acl/acl2020.html'),
      ('ACL2021', 'https://dblp.org/db/conf/acl/acl2021-1.html'),
      ('ACL2022', 'https://dblp.org/db/conf/acl/acl2022-1.html'),
      ('ACL2023', 'https://dblp.org/db/conf/acl/acl2023-1.html'),
      ('ACL2024', 'https://dblp.org/db/conf/acl/acl2024-1.html'),
      ('ACL2025', 'https://dblp.org/db/conf/acl/acl/acl2025-1.html'),
      ('EMNLP2019', 'https://dblp.org/db/conf/emnlp/emnlp2019-1.html'),
      ('EMNLP2020', 'https://dblp.org/db/conf/emnlp/emnlp2020-1.html'),
      ('EMNLP2021', 'https://dblp.org/db/conf/emnlp/emnlp2021.html'),
      ('EMNLP2022', 'https://dblp.org/db/conf/emnlp/emnlp2022.html'),
      ('EMNLP2023', 'https://dblp.org/db/conf/emnlp/emnlp2023.html'),
      ('EMNLP2024', 'https://dblp.org/db/conf/emnlp/emnlp2024.html'),
      ('TMM2019', 'https://dblp.org/db/journals/tmm/tmm21.html'),
      ('TMM2020', 'https://dblp.org/db/journals/tmm/tmm22.html'),
      ('TMM2021', 'https://dblp.org/db/journals/tmm/tmm23.html'),
      ('TMM2022', 'https://dblp.org/db/journals/tmm/tmm24.html'),
      ('TMM2023', 'https://dblp.org/db/journals/tmm/tmm25.html'),
      ('TMM2024', 'https://dblp.org/db/journals/tmm/tmm26.html'),
      ('TMM2025', 'https://dblp.org/db/journals/tmm/tmm27.html'),
      ('WWW2019', 'https://dblp.org/db/conf/www/www2019.html'),
      ('WWW2020', 'https://dblp.org/db/conf/www/www2020.html'),
      ('WWW2021', 'https://dblp.org/db/conf/www/www2021.html'),
      ('WWW2022', 'https://dblp.org/db/conf/www/www2022.html'),
      ('WWW2023', 'https://dblp.org/db/conf/www/www2023.html'),
      ('WWW2024', 'https://dblp.org/db/conf/www/www2024.html'),
      ('WWW2025', 'https://dblp.org/db/conf/www/www2025.html'),
    ]
    sources += generate_dblp_sources('CIKM', 2019, 2025)
    sources += generate_dblp_sources('SIGIR', 2019, 2025)
    sources += generate_dblp_sources('KDD', 2019, 2024)
    sources += generate_dblp_sources('MM', 2019, 2024)
    sources += generate_dblp_sources('CVPR', 2019, 2025)
    sources += generate_dblp_sources('IJCAI', 2019, 2025)
    sources += generate_dblp_sources('AAAI', 2019, 2025)
    sources += generate_dblp_sources('ICML', 2019, 2025)
    sources += generate_dblp_sources('ICLR', 2019, 2025)
    return sources

def run_crawler(papers_dir=PAPERS_DIR, timeout=15):
    os.makedirs(papers_dir, exist_ok=True)
    for name, url in get_sources():
        out_path = os.path.join(papers_dir, f'{name}.txt')
        if os.path.exists(out_path):
            print(f'跳过 {name}: {out_path} 已存在')
            continue
        print(f'name:{name}\turl:{url}')
        try:
            r = requests.get(url, headers={
                'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
            }, timeout=timeout)
        except requests.RequestException as e:
            print(f'访问失败: {url}，错误: {e}')
            continue
        if not r.ok:
            print(f'访问失败: {url} (状态码 {r.status_code})')
            continue
        soup = BeautifulSoup(r.text, 'html.parser')
        page_titles = []
        for paper_item in soup.find_all(class_='title', itemprop='name'):
            try:
                items = [item.string if item.string else item for item in paper_item.contents]
                paper = ''.join([item for item in items if isinstance(item, str)])
            except Exception:
                continue
            page_titles.append(paper)
        existing = set()
        if os.path.exists(out_path):
            try:
                with open(out_path, 'r', encoding='utf-8') as rf:
                    existing = {line.rstrip('') for line in rf}
            except Exception:
                existing = set()
        new_lines = [t for t in page_titles if t not in existing]
        if new_lines:
            with open(out_path, 'a', encoding='utf-8') as wf:
                for t in new_lines:
                    wf.write(f'{t}\n')
            print(f'已写入 {len(new_lines)} 条到 {out_path}')
        else:
            print(f'{name} 本页无新增条目，已跳过写入')

print('爬虫可用。使用 run_crawler() 运行。')

下面两行代码根据需要运行

In [ ]:
run_crawler() # 需要爬取数据

In [ ]:
!unzip papers.zip # 直接解压缩

In [ ]:
!zip -r papers.zip papers # 压缩为 zip

In [ ]:
# 语义检索逻辑（含离线与缓存目录支持）
import glob
import numpy as np
from typing import List, Tuple
from sentence_transformers import SentenceTransformer


def load_titles(papers_dir: str) -> Tuple[List[str], List[str]]:
    titles: List[str] = []
    sources: List[str] = []
    for path in sorted(glob.glob(os.path.join(papers_dir, '*.txt'))):
        conf = os.path.splitext(os.path.basename(path))[0]
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    t = line.strip()
                    if t:
                        titles.append(t)
                        sources.append(conf)
        except Exception:
            continue
    seen = set()
    dedup_titles: List[str] = []
    dedup_sources: List[str] = []
    for t, s in zip(titles, sources):
        if t in seen:
            continue
        seen.add(t)
        dedup_titles.append(t)
        dedup_sources.append(s)
    return dedup_titles, dedup_sources

def _get_model(model_name: str, model_cache_dir: str = None, offline: bool = False):
    if offline:
        os.environ['HF_HUB_OFFLINE'] = '1'
        os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
        os.environ.setdefault('SENTENCE_TRANSFORMERS_HOME', model_cache_dir or os.environ.get('SENTENCE_TRANSFORMERS_HOME', ''))
    kwargs = {}
    if model_cache_dir:
        kwargs['cache_folder'] = model_cache_dir
        os.environ.setdefault('HF_HOME', model_cache_dir)
    return SentenceTransformer(model_name, **kwargs)

def build_or_load_index(model_name: str, papers_dir: str, cache_path: str = None, force_rebuild: bool = False, model_cache_dir: str = None, offline: bool = False):
    titles, sources = load_titles(papers_dir)
    if cache_path is None:
        cache_path = os.path.join(papers_dir, f"_index_{model_name.replace('/', '_')}.npz")
    if (not force_rebuild) and os.path.exists(cache_path):
        try:
            data = np.load(cache_path, allow_pickle=True)
            cached_model = str(data['model_name']) if 'model_name' in data.files else None
            if cached_model == model_name:
                embeddings = data['embeddings']
                titles = list(data['titles'])
                sources = list(data['sources'])
                return titles, sources, embeddings, cache_path
        except Exception:
            pass
    model = _get_model(model_name, model_cache_dir=model_cache_dir, offline=offline)
    embeddings = model.encode(titles, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
    np.savez_compressed(cache_path, model_name=model_name, embeddings=embeddings, titles=np.array(titles, dtype=object), sources=np.array(sources, dtype=object))
    return titles, sources, embeddings, cache_path

def search(query: str, titles: List[str], sources: List[str], embeddings: np.ndarray, model_name: str, top_k: int = 20, model_cache_dir: str = None, offline: bool = False):
    model = _get_model(model_name, model_cache_dir=model_cache_dir, offline=offline)
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = embeddings @ q_emb[0]
    idx = np.argsort(-scores)[:top_k]
    return [(float(scores[i]), titles[i], sources[i]) for i in idx]

print('检索模块已加载。')

In [ ]:
# 运行检索：构建/加载索引并查询

MODEL = 'Qwen/Qwen3-Embedding-0.6B'  #@param {type:"string"}
TOP_K = 50  #@param {type:"number"}
FORCE_REBUILD = False  #@param {type:"boolean"}
QUERY = 'Multimodal LLMs for Recommender Systems'  #@param {type:"string"}

titles, sources, embeddings, cache_path = build_or_load_index(
    MODEL, PAPERS_DIR, None, FORCE_REBUILD, MODEL_CACHE_DIR or None, OFFLINE
)
print(f'索引构建完成: {len(titles)} 条目, 缓存文件: {cache_path}')

results = search(QUERY, titles, sources, embeddings, MODEL, TOP_K, MODEL_CACHE_DIR or None, OFFLINE)
print(f'Top-{TOP_K} 结果:')
for score, title, conf in results:
    print(f'[{conf}] {title}   (cos={score:.4f})')